# LSTM для классификации текстов

## 1. Загрузка необходимых библиотек и данных

### 1.1 Импорты

In [1]:
import re
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.utils.rnn as rnn_utils

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from collections import Counter
from datasets import load_dataset
from sklearn.datasets import fetch_20newsgroups

import nltk
from nltk.corpus import stopwords

In [2]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### 1.2 Emotion

In [3]:
dataset_emotion = load_dataset('emotion', split='train+test')

print(f"emotion size: {len(dataset_emotion)}")

emotion size: 18000


### 1.3 News

In [4]:
categories = ['comp.sys.ibm.pc.hardware',
              'comp.sys.mac.hardware',
              'comp.graphics',
              'comp.windows.x']

dataset_news = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

print(f"news train size: {len(dataset_news.data)}")

news train size: 3906


## 2. Параметры

In [5]:
torch.manual_seed(42)
np.random.seed(42)

In [6]:
EMBEDDING_DIM = 100
HIDDEN_DIM = 64
NUM_LAYERS = 3
DROPOUT = 0.2
BATCH_SIZE = 64
NUM_EPOCHS = 15
LEARNING_RATE = 0.001
MAX_LEN = {
    'emotion': 100,
    'news': 300
}
TOP_K = 20000

In [7]:
if torch.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(device)

cpu


## 3. Предобработка

### 3.1 Удаление шума(для news)

In [8]:
def clean_noise(text):
    text = re.sub(r'^(From|Subject|Organization|Lines|Distribution|NNTP-Posting-Host|Keywords|Summary):.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        if len(line) == 0:
            continue
        non_alnum = sum(1 for c in line if not c.isalnum() and not c.isspace())
        if non_alnum/len(line) > 0.7 and len(line) > 10:
            continue
        cleaned_lines.append(line)
    text = '\n'.join(cleaned_lines)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r"[^\w\s.!?]", '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

### 3.2 Токенизация

In [9]:
stop_words = set(stopwords.words("english"))

def tokenize_text(text):
    words = nltk.word_tokenize(text.lower())
    words = [w for w in words if w not in stop_words]
    return words

In [10]:
preprocessors= {
    'tokenization': tokenize_text
}

### 3.3 Применение предобработки

In [11]:
datasets = {'emotion': dataset_emotion, 'news': dataset_news}
datasets_texts = {}
datasets_labels = {}

In [12]:
datasets_texts['emotion'] = datasets['emotion']['text']
datasets_texts['news'] = [clean_noise(text) for text in datasets['news'].data]

In [13]:
datasets_labels['emotion'] = datasets['emotion']['label']
datasets_labels['news'] = datasets['news'].target

In [14]:
preprocessed_datasets = {'emotion': {}, 'news': {}}

In [15]:
for prep_name, prep_func in preprocessors.items():
    for dataset_name in datasets.keys():
        print(f"Processing {dataset_name} with {prep_name}")
        preprocessed_datasets[dataset_name][prep_name] = [prep_func(text) for text in datasets_texts[dataset_name]]

Processing emotion with tokenization
Processing news with tokenization


## 3.4 Построение словаря

In [16]:
counters = {}
word2idx = {}
X_data = {}
y_data = {}

for dataset_name in datasets.keys():
    counters[dataset_name] = {}
    word2idx[dataset_name] = {}
    X_data[dataset_name] = {}
    y_data[dataset_name] = {}

    for prep_name in preprocessors.keys():
        print(f"Processing {prep_name} with {dataset_name}")

        texts = preprocessed_datasets[dataset_name][prep_name]
        labels = datasets_labels[dataset_name]

        # 1. Строим частотный словарь по всем текстам
        counter = Counter()
        for text in texts:
            counter.update(text)
        counters[dataset_name][prep_name] = counter

        # 2. Создаём word2idx (TOP_K самых частых слов + OOV)
        most_common = counter.most_common(TOP_K-1)
        w2i = {word: i+1 for i, (word, _) in enumerate(most_common)}
        w2i['<OOV>'] = len(w2i)+1
        word2idx[dataset_name][prep_name] = w2i
        vocab_size = len(w2i)+1  # +1 для padding=0
        print(f"vocab size: {vocab_size}")

        # 3. Преобразуем тексты в последовательности индексов
        sequences = []
        for text in texts:
            swq = [w2i.get(w,w2i['<OOV>']) for w in text]
            sequences.append(swq)

        # 4. Паддинг до MAX_LEN[dataset_name]
        max_len = MAX_LEN[dataset_name]
        padded = []

        for seq in sequences:
            if len(seq) >= max_len:
                padded.append(seq[:max_len])
            else:
                padded.append(seq + [0] * (max_len-len(seq)))
        # 5. Преобразуем в тензоры PyTorch
        X_data[dataset_name][prep_name] = torch.tensor(padded)
        y_data[dataset_name][prep_name] = torch.tensor(labels)

Processing tokenization with emotion
vocab size: 16035
Processing tokenization with news
vocab size: 20001


## 4. Классы для LSTM

In [17]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]

        # считаем реальную длину (без padding=0)
        length = max(1, (x != 0).sum().item())

        return x, length, y

    def __len__(self):
        return len(self.X)

In [18]:
def collate_fn(batch):
    texts, lengths, labels = zip(*batch)

    texts = torch.stack(texts)
    lengths = torch.tensor(lengths)
    labels = torch.tensor(labels)

    return texts, lengths, labels

In [19]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, num_layers=2, dropout=0.3):
        super().__init__()

        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x, lengths):
        embedded = self.embedding(x)

        # pack_padded_sequence обрабатывает только lengths токенов в embedded
        # а в lengths длина без паддингов, то есть паддинги не обрабатываем
        packed = rnn_utils.pack_padded_sequence(
            embedded,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.lstm(packed)

        # последние слои двух направлений
        hidden_forward = hidden[self.num_layers-1]
        hidden_backward = hidden[-1]

        hidden_cat = torch.cat((hidden_forward, hidden_backward), dim=1)

        return self.fc(hidden_cat)

## 5. Функция обучения

In [20]:
def train(X_train, y_train, X_test, y_test,
          vocab_size, num_classes,
          emb_dim=EMBEDDING_DIM,
          hid_dim=HIDDEN_DIM,
          n_layers=NUM_LAYERS,
          dropout=DROPOUT,
          batch_size=BATCH_SIZE,
          epochs=NUM_EPOCHS,
          lr=LEARNING_RATE):

    model = LSTMClassifier(
        vocab_size, emb_dim, hid_dim, num_classes,
        num_layers=n_layers, dropout=dropout
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_dataset = TextDataset(X_train, y_train)
    test_dataset = TextDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=collate_fn)

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for texts, lengths, labels in train_loader:
            texts = texts.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(texts, lengths)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * texts.size(0)

        avg_loss = total_loss / len(X_train)

        # --- evaluation ---
        model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for texts, lengths, labels in test_loader:
                texts = texts.to(device)
                lengths = lengths.to(device)
                labels = labels.to(device)

                outputs = model(texts, lengths)
                preds = outputs.argmax(1)

                all_preds.append(preds)
                all_labels.append(labels)

        all_preds = torch.cat(all_preds)
        all_labels = torch.cat(all_labels)

        acc = (all_preds == all_labels).float().mean().item()

        print(f"Epoch {epoch+1}, loss {avg_loss:.4f}, test acc {acc:.4f}")

    f1 = f1_score(all_labels.numpy(), all_preds.numpy(), average='micro')
    print(f"f1-micro: {f1:.4f}")

    return model

## 6. Работа модели

In [21]:
results = []

for dataset_name in datasets.keys():
    for preprocessor_name in preprocessors.keys():
        X = X_data[dataset_name][preprocessor_name]
        y = y_data[dataset_name][preprocessor_name]

        X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=42, stratify=y)

        vocab_size = len(word2idx[dataset_name][preprocessor_name]) + 1
        num_classes = len(torch.unique(y_train))


        model = train(X_train,y_train,X_test,y_test,
                      vocab_size=vocab_size, num_classes=num_classes)

        test_dataset = TextDataset(X_test, y_test)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

        model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for texts, lengths, labels in test_loader:
                texts = texts.to(device)
                lengths = lengths.to(device)

                outputs = model(texts, lengths)
                preds = outputs.argmax(1)

                all_preds.append(preds)
                all_labels.append(labels)

        all_preds = torch.cat(all_preds)
        all_labels = torch.cat(all_labels)

        acc = (all_preds == all_labels).float().mean().item()
        f1_micro = f1_score(all_labels.numpy(), all_preds.numpy(), average='micro')

        results.append({
            'Dataset': dataset_name,
            'Preprocessing': preprocessor_name,
            'Test Accuracy': acc,
            'F1-micro': f1_micro,
            'Vocabulary size': vocab_size,
            'Num classes': num_classes
        })

Epoch 1, loss 1.4062, test acc 0.6528
Epoch 2, loss 0.5926, test acc 0.8350
Epoch 3, loss 0.2653, test acc 0.8664
Epoch 4, loss 0.1587, test acc 0.8767
Epoch 5, loss 0.1133, test acc 0.8783
Epoch 6, loss 0.0807, test acc 0.8842
Epoch 7, loss 0.0690, test acc 0.8939
Epoch 8, loss 0.0498, test acc 0.8922
Epoch 9, loss 0.0471, test acc 0.8872
Epoch 10, loss 0.0397, test acc 0.8839
Epoch 11, loss 0.0304, test acc 0.8939
Epoch 12, loss 0.0256, test acc 0.8969
Epoch 13, loss 0.0263, test acc 0.8861
Epoch 14, loss 0.0343, test acc 0.8842
Epoch 15, loss 0.0294, test acc 0.8939
f1-micro: 0.8939
Epoch 1, loss 1.3791, test acc 0.3338
Epoch 2, loss 1.2066, test acc 0.5294
Epoch 3, loss 0.8238, test acc 0.6304
Epoch 4, loss 0.5077, test acc 0.6841
Epoch 5, loss 0.3215, test acc 0.6893
Epoch 6, loss 0.1987, test acc 0.7008
Epoch 7, loss 0.1204, test acc 0.6944
Epoch 8, loss 0.0738, test acc 0.6982
Epoch 9, loss 0.0492, test acc 0.7148
Epoch 10, loss 0.0341, test acc 0.7225
Epoch 11, loss 0.0252, tes

## 7. Результаты

In [23]:
df_results = pd.DataFrame(results)
display(df_results)

,Dataset,Preprocessing,Test Accuracy,F1-micro,Vocabulary size,Num classes
0,emotion,tokenization,0.893889,0.893889,16035,6
1,news,tokenization,0.714834,0.714834,20001,4
